In [0]:
from pyspark.sql.functions import col, avg, min, count

# 1. Read the enriched Silver data
# We are using the laps table because it has the names and the timing data
silver_laps_df = spark.read.table("f1_project.silver.enriched_laps")

# 2. Create the "Driver Performance Summary"
# We filter out pit out laps because they are naturally much slower
gold_performance_df = (silver_laps_df
    .filter(col("is_pit_out_lap") == False)
    .groupBy("team_name", "full_name")
    .agg(
        min("lap_duration").alias("fastest_lap_seconds"),
        avg("lap_duration").alias("average_lap_time"),
        count("lap_number").alias("total_valid_laps")
    )
    .orderBy(col("fastest_lap_seconds").asc())
)

# 3. Save to Gold Table
(gold_performance_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("f1_project.gold.driver_performance_metrics"))

print("Gold Layer Updated! Driver Leaderboard is ready.")
display(spark.table("f1_project.gold.driver_performance_metrics"))